In [ ]:
import torch
import torch.nn as nn
from torchvision import transforms, models
from PIL import Image
import os
import torch.nn.functional as F
import json

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_PATH = "Best_Model.pth"
IMG_DIR = "../data/finaltestimages"
LABELS_JSON = "../data/labels/final_test_map.json"
RESULTS_JSON = "results.json"

CATEGORIES = ["Słupy Elektryczne", "Uszkodzona droga", "Uszkodzony Znak", "Powalone drzewa", "Smieci", "Graffiti"]

TRANSLATE_MAP = {
    "FallenTrees": "Powalone drzewa",
    "Graffitti": "Graffiti",
    "DamagedRoad": "Uszkodzona droga",
    "Garbage": "Smieci",
    "DamagedRoadSigns": "Uszkodzony Znak",
    "DamagedElectricalPoles": "Słupy Elektryczne"
}

test_transforms = transforms.Compose([
    transforms.Resize((624, 624)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

def load_model(path, num_classes):
    model = models.resnet18(weights=None)
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, num_classes)
    model.load_state_dict(torch.load(path, map_location=DEVICE))
    model.to(DEVICE)
    model.eval()
    return model

def run_test():
    if not os.path.exists(IMG_DIR):
        print(f"Błąd: Katalog {IMG_DIR} nie istnieje!")
        return
    if not os.path.exists(LABELS_JSON):
        print(f"Błąd: Plik etykiet {LABELS_JSON} nie istnieje!")
        return

    model = load_model(MODEL_PATH, len(CATEGORIES))
    
    with open(LABELS_JSON, 'r', encoding='utf-8') as f:
        true_labels_map = json.load(f)

    print(f"Znaleziono {len(true_labels_map)} zdjęć do zweryfikowania.\n")
    print(f"{'Plik':<25} | {'Przewidywana':<18} | {'Prawdziwa':<18} | {'Pewność'}")
    print("-" * 90)

    results_details = []
    correct_count = 0

    with torch.no_grad():
        for img_name, raw_true_class in true_labels_map.items():
            img_path = os.path.join(IMG_DIR, img_name)
            
            if not os.path.exists(img_path):
                continue
            
            true_class = TRANSLATE_MAP.get(raw_true_class, raw_true_class)

            img = Image.open(img_path).convert('RGB')
            img_tensor = test_transforms(img).unsqueeze(0).to(DEVICE)

            outputs = model(img_tensor)
            probabilities = F.softmax(outputs, dim=1)[0]
            prob_percent, class_idx = torch.max(probabilities, 0)

            predicted_category = CATEGORIES[class_idx]
            confidence = prob_percent.item() * 100
            
            is_correct = (predicted_category == true_class)
            if is_correct:
                correct_count += 1

            prob_dist = {CATEGORIES[i]: f"{probabilities[i].item()*100:.1f}%" for i in range(len(CATEGORIES))}
            
            status = "OK" if is_correct else "BŁĄD"
            print(f"{img_name:<25} | {predicted_category:<18} | {true_class:<18} | {confidence:>6.2f}% [{status}]")
            print(f"   ∟ Rozkład: {prob_dist}\n")

            results_details.append({
                "plik": img_name,
                "przewidywana": predicted_category,
                "prawdziwa": true_class,
                "pewnosc": f"{confidence:.2f}%",
                "poprawnie": is_correct,
                "rozklad": prob_dist
            })

    total = len(results_details)
    accuracy = (correct_count / total) * 100 if total > 0 else 0
    
    print("-" * 90)
    print(f"SKUTECZNOŚĆ KOŃCOWA: {accuracy:.2f}% ({correct_count}/{total})")

    final_report = {
        "summary": {
            "accuracy": f"{accuracy:.2f}%",
            "correct": correct_count,
            "total": total
        },
        "results": results_details
    }

    with open(RESULTS_JSON, 'w', encoding='utf-8') as f:
        json.dump(final_report, f, indent=4, ensure_ascii=False)
    
    print(f"Szczegółowy raport zapisany w {RESULTS_JSON}")

if __name__ == "__main__":
    run_test()

C:\Users\mkuzm\AppData\Local\Temp\ipykernel_6200\1069959611.py:40: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(path, map_location=DEVICE))

Znaleziono 20 zdjęć do zweryfikowania.

Plik                      | Przewidywana       | Prawdziwa          | Pewność
------------------------------------------------------------------------------------------
20260408_201438.jpg       | Słupy Elektryczne  | Powalone drzewa    |  39.68% [BŁĄD]
   ∟ Rozkład: {'Słupy Elektryczne': '39.7%', 'Uszkodzona droga': '16.5%', 'Uszkodzony Znak': '6.0%', 'Powalone drzewa': '11.0%', 'Smieci': '13.2%', 'Graffiti': '13.6%'}

20260411_150749.jpg       | Graffiti           | Graffiti           |  70.03% [OK]
   ∟ Rozkład: {'Słupy Elektryczne': '9.3%', 'Uszkodzona droga': '1.6%', 'Uszkodzony Znak': '7.1%', 'Powalone drzewa': '1.8%', 'Smieci': '10.2%', 'Graffiti': '70.0%'}

20260411_151205.jpg       | Graffiti           | Graffiti           |  85.36% [OK]
   ∟ Rozkład: {'Słupy Elektryczne': '3.0%', 'Uszkodzona droga': '2.1%', 'Uszkodzony Znak': '3.4%', 'Powalone drzewa': '1.7%', 'Smieci': '4.4%', 'Graffiti': '85.4%'}

20260413_161528.jpg       | Uszkodzon